In [1]:
import sys
from pathlib import Path
import pandas as pd

src_dir = Path.cwd().parent

# sys.path strictly for importing modules
sys.path.append(str(src_dir))
from utils.data_utils import *

HOSP_DIR = src_dir / "data" / "mimic-iv" / "hosp"

In [2]:
diabetic_patients = load_data(src_dir / "data" / "processed" / "diabetic_patients.csv.gz")
diabetic_patients.head()

,subject_id,hadm_id,seq_num,icd_code,icd_version
0,10000635,20642640,7,E119,10
1,10000980,20897796,5,E1122,10
2,10001176,23334588,2,25000,9
3,10001843,21728396,9,E119,10
4,10001877,21320596,6,25000,9


In [3]:
diabetic_ids = diabetic_patients["subject_id"].unique()
diabetic_ids

array(['10000635', '10000980', '10001176', ..., '19999287', '19999379',
       '19999828'], dtype=object)

In [4]:
# Define all file paths for cohort extraction
patients_path = HOSP_DIR / "patients.csv.gz"
admissions_path = HOSP_DIR / "admissions.csv.gz"
labevents_path = HOSP_DIR / "labevents.csv.gz"
prescriptions_path = HOSP_DIR / "prescriptions.csv.gz"
diagnoses_icd_path = HOSP_DIR / "diagnoses_icd.csv.gz"
procedures_icd_path = HOSP_DIR / "procedures_icd.csv.gz"

In [6]:
# Load data
patients = load_data(patients_path)
admissions = load_data(admissions_path)
diagnoses_icd = load_data(diagnoses_icd_path)
procedures_icd = load_data(procedures_icd_path)

# lab_usecols = ["subject_id", "hadm_id", "itemid", "charttime", "valuenum"]
# labevents = load_data(labevents_path, usecols=lab_usecols)

labevents_filtered = []
for chunk in pd.read_csv(labevents_path, compression="gzip", chunksize=500000,
                         low_memory=False, dtype=str):
    filtered = chunk[chunk["subject_id"].isin(diabetic_ids)]
    labevents_filtered.append(filtered)

labevents = pd.concat(labevents_filtered)

# pres_usecols = ["subject_id", "hadm_id", "drug", "starttime", "stoptime"]
# prescriptions = load_data(prescriptions_path, usecols=pres_usecols)

# Filter data for diabetic patients
patients = filter_to_cohort(patients, diabetic_ids)
admissions = filter_to_cohort(admissions, diabetic_ids)
diagnoses_icd = filter_to_cohort(diagnoses_icd, diabetic_ids)
procedures_icd = filter_to_cohort(procedures_icd, diabetic_ids)
labevents = filter_to_cohort(labevents, diabetic_ids)
# prescriptions = filter_to_cohort(prescriptions, diabetic_ids)

In [9]:
procedures_icd.head()

,subject_id,hadm_id,seq_num,chartdate,icd_code,icd_version
7,10000635,26134563,1,2136-06-19,3734,9
8,10000635,26134563,2,2136-06-19,3728,9
9,10000635,26134563,3,2136-06-19,3727,9
25,10000980,25242409,1,2191-04-05,4513,9
26,10000980,26913865,1,2189-06-30,0066,9


In [10]:
def create_condition_flag(diagnoses_icd, code_prefixes, flag_name):
    matched = diagnoses_icd[
        diagnoses_icd["icd_code"].str.startswith(tuple(code_prefixes))
    ]

    flag = (
        matched.groupby("subject_id")
        .size()
        .gt(0)
        .rename(flag_name)
        .reset_index()
    )
    return flag


In [12]:
patients_summary = patients[["subject_id", "gender", "anchor_age"]].rename(columns={"anchor_age": "age"})
admissions_summary = (
    admissions.groupby("subject_id")
    .agg(
        n_admissions=("hadm_id", "nunique"),
        first_admission_date=("admittime", "min"),
        last_admission_date=("admittime", "max"),
    )
    .reset_index()
)
HTN_CODES = ["I10", "401"]
CKD_CODES = ["N18", "585"]
OBESITY_CODES = ["E66", "2780"]
NEUROPATHY_CODES = ["E104", "E114", "G60", "G62", "2506", "3572"]
RETINOPATHY_CODES = ["E103", "E113", "H35", "3620"]
HEART_DISEASE_CODES = ["I2", "I42", "I50", "41", "425", "428"]

htn_flag = create_condition_flag(diagnoses_icd, HTN_CODES, "hypertension_flag")
ckd_flag = create_condition_flag(diagnoses_icd, CKD_CODES, "ckd_flag")
obesity_flag = create_condition_flag(diagnoses_icd, OBESITY_CODES, "obesity_flag")
neuropathy_flag = create_condition_flag(diagnoses_icd, NEUROPATHY_CODES, "neuropathy_flag")
retinopathy_flag = create_condition_flag(diagnoses_icd, RETINOPATHY_CODES, "retinopathy_flag")
heart_flag = create_condition_flag(diagnoses_icd, HEART_DISEASE_CODES, "heart_disease_flag")


obesity_flag["obesity_flag"].value_counts()

obesity_flag
True    11842
Name: count, dtype: int64

In [ ]:
cohort = (
    patients_summary
    .merge(admissions_summary, on="subject_id", how="left")
    .merge(htn_flag, on="subject_id", how="left")
    .merge(ckd_flag, on="subject_id", how="left")
    .merge(obesity_flag, on="subject_id", how="left")
    .merge(neuropathy_flag, on="subject_id", how="left")
    .merge(retinopathy_flag, on="subject_id", how="left")
    .merge(heart_flag, on="subject_id", how="left")
)